# CNN Take 4 Explained

This notebook explains the CNN models and training process used in this project, ending with **CNN Take 4**, the best run so far.

If links in chat were not opening, use the file links inside this notebook or open the files directly from the project tree.

In [ ]:
%matplotlib inline

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "Notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from SRC.cnn_model import SimpleCIFAR10CNN
from SRC.improved_cnn import DeeperCIFAR10CNN

BASELINE_PATH = PROJECT_ROOT / "Data" / "cnn_baseline_outputs" / "run_20260325_130955" / "summary.json"
IMPROVED_PATH = PROJECT_ROOT / "Data" / "improved_cnn_outputs" / "run_20260326_151826" / "summary.json"
UPGRADED_PATH = PROJECT_ROOT / "Data" / "improved_cnn_outputs" / "run_20260326_161003" / "summary.json"
TAKE4_PATH = PROJECT_ROOT / "Data" / "improved_cnn_outputs" / "cnn_take_4_20260326_234323" / "summary.json"

with BASELINE_PATH.open("r", encoding="utf-8") as handle:
    baseline = json.load(handle)
with IMPROVED_PATH.open("r", encoding="utf-8") as handle:
    improved = json.load(handle)
with UPGRADED_PATH.open("r", encoding="utf-8") as handle:
    upgraded = json.load(handle)
with TAKE4_PATH.open("r", encoding="utf-8") as handle:
    take4 = json.load(handle)

print(TAKE4_PATH)

## 1. What a CNN Learns

A CNN learns features in stages.

- Early layers learn low-level patterns such as edges, corners, and color changes.
- Middle layers learn textures, repeated shapes, and local object parts.
- Deeper layers combine those patterns into higher-level class information.

For CIFAR-10, this matters because the images are very small (`32 x 32`), so the model has to use its spatial information carefully.

In [ ]:
baseline_model = SimpleCIFAR10CNN(dropout=baseline["dropout"])
take4_model = DeeperCIFAR10CNN(
    channels=tuple(take4["channels"]),
    blocks_per_stage=tuple(take4["blocks_per_stage"]),
    pooling_type=take4["pooling_type"],
    dropout=take4["dropout"],
    classifier_hidden=take4["classifier_hidden"],
)

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

architecture_table = pd.DataFrame(
    [
        {
            "model": "First CNN baseline",
            "feature_channels": "32, 64, 128",
            "blocks_per_stage": "2, 2, 2",
            "pooling": "max",
            "batch_norm": "no",
            "trainable_parameters": count_parameters(baseline_model),
        },
        {
            "model": "CNN Take 4",
            "feature_channels": ", ".join(str(value) for value in take4["channels"]),
            "blocks_per_stage": ", ".join(str(value) for value in take4["blocks_per_stage"]),
            "pooling": take4["pooling_type"],
            "batch_norm": "yes",
            "trainable_parameters": count_parameters(take4_model),
        },
    ]
)

display(architecture_table)
take4_model.describe_feature_shapes()

## 2. How the Training Recipe Improved Over Time

The model did not improve only because it got deeper. The training recipe also became much stronger.

- The baseline had a short, simple training setup.
- The improved model added a better architecture.
- The upgraded runs added stronger augmentation and a better learning-rate schedule.
- CNN Take 4 also used learned stride downsampling and explicit color augmentation.

In [ ]:
training_table = pd.DataFrame(
    [
        {
            "setting": "augmentation policy",
            "baseline": baseline.get("augment"),
            "improved": improved.get("augment"),
            "upgraded": upgraded.get("augmentation_policy"),
            "take4": take4.get("augmentation_policy"),
        },
        {
            "setting": "optimizer",
            "baseline": "adam",
            "improved": improved.get("optimizer"),
            "upgraded": upgraded.get("optimizer"),
            "take4": take4.get("optimizer"),
        },
        {
            "setting": "label smoothing",
            "baseline": 0.0,
            "improved": improved.get("label_smoothing"),
            "upgraded": upgraded.get("label_smoothing"),
            "take4": take4.get("label_smoothing"),
        },
        {
            "setting": "MixUp alpha",
            "baseline": 0.0,
            "improved": improved.get("mixup_alpha", 0.0),
            "upgraded": upgraded.get("mixup_alpha", 0.0),
            "take4": take4.get("mixup_alpha", 0.0),
        },
        {
            "setting": "CutMix alpha",
            "baseline": 0.0,
            "improved": improved.get("cutmix_alpha", 0.0),
            "upgraded": upgraded.get("cutmix_alpha", 0.0),
            "take4": take4.get("cutmix_alpha", 0.0),
        },
        {
            "setting": "scheduler",
            "baseline": "none",
            "improved": "none",
            "upgraded": upgraded.get("scheduler"),
            "take4": take4.get("scheduler"),
        },
        {
            "setting": "pooling type",
            "baseline": "max",
            "improved": improved.get("pooling_type"),
            "upgraded": upgraded.get("pooling_type"),
            "take4": take4.get("pooling_type"),
        },
    ]
)

display(training_table)

take4_recipe = pd.Series(
    {
        "epochs": take4["epochs"],
        "batch_size": take4["batch_size"],
        "learning_rate": take4["learning_rate"],
        "min_learning_rate": take4["min_learning_rate"],
        "dropout": take4["dropout"],
        "random_erasing_prob": take4["random_erasing_prob"],
        "color_jitter_brightness": take4["color_jitter_brightness"],
        "color_jitter_contrast": take4["color_jitter_contrast"],
        "color_jitter_saturation": take4["color_jitter_saturation"],
        "color_jitter_hue": take4["color_jitter_hue"],
        "random_grayscale_prob": take4["random_grayscale_prob"],
    },
    name="cnn_take_4",
)
display(take4_recipe.to_frame())

## 3. Accuracy Progress Across Runs

This is the clearest summary of the project so far: every major upgrade improved the final test accuracy.

In [ ]:
results = pd.DataFrame(
    [
        {"run": "Baseline CNN", "best_val_accuracy": baseline["best_val_accuracy"], "test_accuracy": baseline["test_accuracy"]},
        {"run": "Improved CNN", "best_val_accuracy": improved["best_val_accuracy"], "test_accuracy": improved["test_accuracy"]},
        {"run": "Upgraded CNN", "best_val_accuracy": upgraded["best_val_accuracy"], "test_accuracy": upgraded["test_accuracy"]},
        {"run": "CNN Take 4", "best_val_accuracy": take4["best_val_accuracy"], "test_accuracy": take4["test_accuracy"]},
    ]
)

display(results)

plt.figure(figsize=(9, 5))
plt.bar(results["run"], results["test_accuracy"], color=["#7b8cde", "#58a55c", "#e08b3c", "#d44b4b"])
plt.axhline(0.80, color="black", linestyle="--", linewidth=1, label="80% target")
plt.ylabel("Test Accuracy")
plt.ylim(0.0, 0.9)
plt.title("CNN Accuracy Progress")
plt.legend()
plt.tight_layout()
plt.show()

## 4. CNN Take 4 Training Curve

CNN Take 4 crossed the `80%` target and became the strongest run in the project.

In [ ]:
take4_history = pd.DataFrame(take4["history"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(take4_history["epoch"], take4_history["train_loss"], marker="o", label="Train loss")
axes[0].plot(take4_history["epoch"], take4_history["val_loss"], marker="o", label="Validation loss")
axes[0].set_title("CNN Take 4 Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(take4_history["epoch"], take4_history["train_accuracy"], marker="o", label="Train accuracy")
axes[1].plot(take4_history["epoch"], take4_history["val_accuracy"], marker="o", label="Validation accuracy")
axes[1].axhline(take4["test_accuracy"], color="tab:red", linestyle="--", label=f"Test accuracy ({take4['test_accuracy']:.4f})")
axes[1].set_title("CNN Take 4 Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.0)
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Key Files

Useful project files:

- [Report](./cnn_step_by_step_report.md)
- [Comparison notebook](./improved_cnn_comparison.ipynb)
- [Baseline model](../SRC/cnn_model.py)
- [Improved trainer](../SRC/train_improved_cnn.py)
- [Take 4 summary](../Data/improved_cnn_outputs/cnn_take_4_20260326_234323/summary.json)
- [Take 4 training curve image](../Data/improved_cnn_outputs/cnn_take_4_20260326_234323/training_curves.png)

Plain path if you want to open the final run directly:

`Data/improved_cnn_outputs/cnn_take_4_20260326_234323/summary.json`